In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
DATA_PATH = "../data/raw/"
os.listdir(DATA_PATH)

In [ ]:
books = pd.read_csv(DATA_PATH + "Books.csv")
users = pd.read_csv(DATA_PATH + "Users.csv")
ratings = pd.read_csv(DATA_PATH + "Ratings.csv")
google_books = pd.read_csv(DATA_PATH + "google_books_dataset.csv")

In [ ]:
print("Books Shape:", books.shape)
print("Users Shape:", users.shape)
print("Ratings Shape:", ratings.shape)
print("Google Books Shape:", google_books.shape)

In [ ]:
print("BOOKS COLUMNS")
print(books.columns.tolist())

print("\nUSERS COLUMNS")
print(users.columns.tolist())

print("\nRATINGS COLUMNS")
print(ratings.columns.tolist())

print("\nGOOGLE BOOKS COLUMNS")
print(google_books.columns.tolist())

In [ ]:
books.head()


In [ ]:
google_books.head()

In [ ]:
print("Books ISBN dtype:")
print(books["ISBN"].dtype)

print("\nGoogle ISBN-10 dtype:")
print(google_books["isbn_10"].dtype)

print("\nGoogle ISBN-13 dtype:")
print(google_books["isbn_13"].dtype)

In [ ]:
print("Missing ISBN-10:")
print(google_books["isbn_10"].isna().sum())

print("\nMissing ISBN-13:")
print(google_books["isbn_13"].isna().sum())

In [ ]:
print("Book-Crossing ISBN Examples")
print(books["ISBN"].head())

print("\nGoogle ISBN-10 Examples")
print(google_books["isbn_10"].head())

print("\nGoogle ISBN-13 Examples")
print(google_books["isbn_13"].head())

In [ ]:
bx_isbn = set(books["ISBN"])

google_isbn10 = set(
    google_books["isbn_10"].dropna()
)

In [ ]:
common_isbn = bx_isbn.intersection(
    google_isbn10
)

print("Matching ISBNs:", len(common_isbn))

In [ ]:
books[
    ["Book-Title", "Book-Author"]
].head(10)

In [ ]:
google_books[
    ["title", "authors"]
].head(10)

In [ ]:
bx_titles = set(
    books["Book-Title"]
    .str.lower()
    .str.strip()
)

google_titles = set(
    google_books["title"]
    .str.lower()
    .str.strip()
    .dropna()
)

In [ ]:
common_titles = bx_titles.intersection(
    google_titles
)

print("Matching Titles:", len(common_titles))

In [ ]:
list(common_titles)[:20]

In [ ]:
books["Book-Author"].sample(10, random_state=42)

In [ ]:
google_books["authors"].sample(10, random_state=42)

In [ ]:
title_matches = books[
    books["Book-Title"].str.lower().isin(common_titles)
][["Book-Title", "Book-Author"]]

title_matches.head(20)

In [ ]:
print("Missing Titles:")
print(google_books["title"].isna().sum())

print("\nMissing Authors:")
print(google_books["authors"].isna().sum())

print("\nMissing Descriptions:")
print(google_books["description"].isna().sum())

print("\nMissing Categories:")
print(google_books["categories"].isna().sum())

In [ ]:
import re

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove apostrophes entirely
    text = text.replace("'", "")

    # Replace remaining punctuation with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
examples = [
    "The Kitchen God's Wife",
    "THE FIRM",
    " Harry   Potter ",
    "Nora Roberts!",
    np.nan
]

for x in examples:
    print(x, " --> ", normalize_text(x))

In [ ]:
books["title_clean"] = books["Book-Title"].apply(
    normalize_text
)

google_books["title_clean"] = google_books["title"].apply(
    normalize_text
)

In [ ]:
def extract_primary_author(author_text):

    if pd.isna(author_text):
        return ""

    return str(author_text).split(",")[0]

In [ ]:
books["author_clean"] = books["Book-Author"].apply(
    normalize_text
)

google_books["author_clean"] = (
    google_books["authors"]
    .apply(extract_primary_author)
    .apply(normalize_text)
)

In [ ]:
books[
    ["Book-Title", "title_clean",
     "Book-Author", "author_clean"]
].head()

In [ ]:
google_books[
    ["title", "title_clean",
     "authors", "author_clean"]
].head()

In [ ]:
books["match_key"] = (
    books["title_clean"]
    + "_"
    + books["author_clean"]
)

google_books["match_key"] = (
    google_books["title_clean"]
    + "_"
    + google_books["author_clean"]
)

In [ ]:
bx_keys = set(books["match_key"])

google_keys = set(google_books["match_key"])

common_keys = bx_keys.intersection(google_keys)

print("Matching Title+Author Keys:", len(common_keys))

In [ ]:
matched_books = books[
    books["match_key"].isin(common_keys)
][[
    "Book-Title",
    "Book-Author",
    "match_key"
]]

matched_books.head(20)

In [ ]:
title_overlap = books.merge(
    google_books,
    on="title_clean",
    suffixes=("_bx", "_google")
)

In [ ]:
title_overlap[
    title_overlap["author_clean_bx"]
    != title_overlap["author_clean_google"]
][[
    "Book-Title",
    "Book-Author",
    "authors"
]].head(20)

In [ ]:
matched_df = books.merge(
    google_books,
    on="match_key",
    how="inner",
    suffixes=("_bx", "_google")
)

print("Matched rows:", matched_df.shape[0])

matched_df[
    [
        "Book-Title",
        "Book-Author",
        "title",
        "authors"
    ]
].head(20)

In [ ]:
google_books["match_key"].duplicated().sum()

In [ ]:
google_books["match_key"].value_counts().head(20)

In [ ]:
google_books["author_clean"].eq("").sum()

In [ ]:
matched_df["author_clean_google"].eq("").sum()

In [ ]:
matched_df["match_key"].value_counts().head(20)

In [ ]:
matched_df["match_key"].nunique()

In [ ]:
print("Matched rows:", matched_df.shape[0])

print("Unique matched books:",
      matched_df["match_key"].nunique())

In [ ]:
google_books["metadata_score"] = (
    google_books["description"].notna().astype(int)
    + google_books["categories"].notna().astype(int)
    + google_books["language"].notna().astype(int)
    + google_books["page_count"].notna().astype(int)
    + google_books["thumbnail"].notna().astype(int)
)

In [ ]:
google_best = (
    google_books
    .sort_values(
        "metadata_score",
        ascending=False
    )
    .drop_duplicates(
        subset="match_key",
        keep="first"
    )
)

In [ ]:
print("Original Google Books:", google_books.shape)

print("After Deduplication:", google_best.shape)

In [ ]:
google_best["match_key"].duplicated().sum()

In [ ]:
google_selected = google_best[
    [
        "match_key",
        "description",
        "categories",
        "language",
        "page_count",
        "thumbnail"
    ]
]

In [ ]:
books_master = books.merge(
    google_selected,
    on="match_key",
    how="left"
)

In [ ]:
print("Books Shape:", books.shape)

print("Books Master Shape:", books_master.shape)

In [ ]:
books_master["description"].notna().sum()


In [ ]:
books_master["categories"].notna().sum()

In [ ]:
enriched_books = books_master[
    books_master["description"].notna()
]

print(enriched_books.shape)

In [ ]:
enriched_books[
    [
        "Book-Title",
        "Book-Author",
        "categories",
        "language"
    ]
].sample(20, random_state=42)

In [ ]:
books_master.columns.tolist()

In [ ]:
books_master = books_master.drop(
    columns=[
        "title_clean",
        "author_clean",
        "match_key"
    ]
)

In [ ]:
ratings.head()

In [ ]:
ratings_master = ratings.rename(
    columns={
        "Book-Rating": "Rating"
    }
)

In [ ]:
books_master.to_csv(
    "../data/processed/books_master.csv",
    index=False
)

ratings_master.to_csv(
    "../data/processed/ratings_master.csv",
    index=False
)

In [ ]:
print(books_master.shape)
print(ratings_master.shape)